In [2]:
# ============================================
# STEP 2: FEATURE COMPATIBILITY ANALYSIS
# ============================================

import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

# Load datasets
dengue = pd.read_csv("/content/drive/My Drive/Dengue_Cleaned1.csv")
typhoid = pd.read_csv("/content/drive/My Drive/typhoid_clean.csv")

print("Dengue shape :", dengue.shape)
print("Typhoid shape:", typhoid.shape)

print("\n" + "="*60)
print("DENGUE COLUMNS")
print("="*60)

for col in dengue.columns:
    print(col)

print("\n" + "="*60)
print("TYPHOID COLUMNS")
print("="*60)

for col in typhoid.columns:
    print(col)

Mounted at /content/drive
Dengue shape : (416, 30)
Typhoid shape: (31087, 23)

DENGUE COLUMNS
Gender
Age
Where_Infected
Year
Course_at_admission_(day)
Fever
Body_Temparature
Fever_Duration
Rash
Course_when_get_Rash
Muscle_Pain
Bleeding
WBC
N
HB
PLT
ALT
AST
BUN
Cr
CK
CK-MB
K+
Na+
C1
CD_4+
CD_8+
CRP
PCT
Hospitalization

TYPHOID COLUMNS
Age
Gender
Location
Socioeconomic Status
Water Source Type
Sanitation Facilities
Hand Hygiene
Consumption of Street Food
Fever Duration (Days)
Gastrointestinal Symptoms
Neurological Symptoms
Skin Manifestations
Complications
White Blood Cell Count
Platelet Count
Blood Culture Result
Widal Test
Typhidot Test
Typhoid Vaccination Status
Previous History of Typhoid
Weather Condition
Ongoing Infection in Society
Typhoid Status


In [3]:
# ============================================
# STEP 3: INSPECT COMMON FEATURE VALUES
# ============================================

features_to_check = {
    "Dengue": [
        "Gender",
        "Age",
        "Fever",
        "Body_Temparature",
        "Fever_Duration",
        "Rash",
        "Muscle_Pain",
        "Bleeding",
        "WBC",
        "PLT"
    ],

    "Typhoid": [
        "Gender",
        "Age",
        "Fever Duration (Days)",
        "Gastrointestinal Symptoms",
        "Neurological Symptoms",
        "Skin Manifestations",
        "White Blood Cell Count",
        "Platelet Count"
    ]
}

print("="*60)
print("DENGUE FEATURE VALUES")
print("="*60)

for col in features_to_check["Dengue"]:
    print(f"\n{col}")
    print(dengue[col].value_counts(dropna=False).head(15))

print("\n" + "="*60)
print("TYPHOID FEATURE VALUES")
print("="*60)

for col in features_to_check["Typhoid"]:
    print(f"\n{col}")
    print(typhoid[col].value_counts(dropna=False).head(15))

DENGUE FEATURE VALUES

Gender
Gender
Male      291
Female    125
Name: count, dtype: int64

Age
Age
30    20
26    19
29    18
33    18
27    17
31    15
28    15
37    14
38    13
45    13
25    13
39    12
36    12
24    12
42    11
Name: count, dtype: int64

Fever
Fever
1    414
0      2
Name: count, dtype: int64

Body_Temparature
Body_Temparature
39      97
39.5    39
38      31
40      30
38.5    24
39.2    17
39.3    15
39.6    14
39.4    13
39.1    13
38.9    11
38.6     9
38.3     9
38.8     9
39.8     9
Name: count, dtype: int64

Fever_Duration
Fever_Duration
4     81
5     75
3     60
6     59
2     41
7     36
1     29
8     14
9      5
10     4
?      3
15     2
20     1
11     1
14     1
Name: count, dtype: int64

Rash
Rash
No     223
Yes    193
Name: count, dtype: int64

Muscle_Pain
Muscle_Pain
Yes    320
No      95
??       1
Name: count, dtype: int64

Bleeding
Bleeding
0    343
1     73
Name: count, dtype: int64

WBC
WBC
2.7     7
1.74    5
2.48    5
4.44    4
3.75    4

In [4]:
# ============================================
# STEP 4: CREATE COMMON IDENTIFICATION FEATURES
# ============================================

import pandas as pd
import numpy as np

# Load original datasets
dengue = pd.read_csv("/content/drive/My Drive/Dengue_Cleaned1.csv")
typhoid = pd.read_csv("/content/drive/My Drive/typhoid_clean.csv")

# ------------------------------------------------
# DENGUE
# ------------------------------------------------

d = pd.DataFrame()

d["Age"] = pd.to_numeric(dengue["Age"], errors="coerce")

d["Gender"] = dengue["Gender"].astype(str).str.strip().str.title()

# Fever: already binary in Dengue
d["Fever"] = pd.to_numeric(dengue["Fever"], errors="coerce")

# Fever duration
d["Fever_Duration"] = pd.to_numeric(
    dengue["Fever_Duration"], errors="coerce"
)

# Rash -> common binary feature
d["Skin_Manifestation"] = (
    dengue["Rash"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"yes": 1, "no": 0})
)

# WBC
d["WBC"] = pd.to_numeric(
    dengue["WBC"], errors="coerce"
)

# Platelets
d["Platelet_Count"] = pd.to_numeric(
    dengue["PLT"], errors="coerce"
)

# Disease label
d["Disease"] = "Dengue"


# ------------------------------------------------
# TYPHOID
# ------------------------------------------------

t = pd.DataFrame()

t["Age"] = pd.to_numeric(
    typhoid["Age"], errors="coerce"
)

t["Gender"] = (
    typhoid["Gender"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Fever from fever duration
t["Fever_Duration"] = pd.to_numeric(
    typhoid["Fever Duration (Days)"],
    errors="coerce"
)

t["Fever"] = (
    t["Fever_Duration"] > 0
).astype(int)

# Skin manifestation -> same binary representation
t["Skin_Manifestation"] = (
    typhoid["Skin Manifestations"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"yes": 1, "no": 0})
)

# WBC
t["WBC"] = pd.to_numeric(
    typhoid["White Blood Cell Count"],
    errors="coerce"
)

# Platelets
t["Platelet_Count"] = pd.to_numeric(
    typhoid["Platelet Count"],
    errors="coerce"
)

# Disease label
t["Disease"] = "Typhoid"


# ------------------------------------------------
# COMBINE
# ------------------------------------------------

common_features = pd.concat(
    [d, t],
    ignore_index=True
)

print("Combined dataset shape:", common_features.shape)

print("\nColumns:")
print(common_features.columns.tolist())

print("\nDisease distribution:")
print(common_features["Disease"].value_counts())

print("\nMissing values:")
print(common_features.isnull().sum())

Combined dataset shape: (31503, 8)

Columns:
['Age', 'Gender', 'Fever', 'Fever_Duration', 'Skin_Manifestation', 'WBC', 'Platelet_Count', 'Disease']

Disease distribution:
Disease
Typhoid    31087
Dengue       416
Name: count, dtype: int64

Missing values:
Age                   0
Gender                0
Fever                 0
Fever_Duration        4
Skin_Manifestation    0
WBC                   1
Platelet_Count        0
Disease               0
dtype: int64


In [5]:
pd.to_numeric(..., errors="coerce")

Ellipsis

In [6]:
# ============================================
# STEP 5: VERIFY COMBINED DATA
# ============================================

print("First 10 rows:")
display(common_features.head(10))

print("\nLast 10 rows:")
display(common_features.tail(10))

print("\nDisease counts:")
print(common_features["Disease"].value_counts())

print("\nData types:")
print(common_features.dtypes)

print("\nMissing values:")
print(common_features.isnull().sum())

First 10 rows:


,Age,Gender,Fever,Fever_Duration,Skin_Manifestation,WBC,Platelet_Count,Disease
0,30,Male,1,6.0,0,8.55,50,Dengue
1,30,Male,1,4.0,0,2.48,187,Dengue
2,29,Male,1,7.0,0,3.39,39,Dengue
3,43,Female,1,2.0,0,2.59,50,Dengue
4,27,Male,1,15.0,1,5.60,270,Dengue
5,29,Male,1,2.0,1,2.10,42,Dengue
6,43,Male,1,4.0,0,1.75,69,Dengue
7,30,Female,1,10.0,1,3.43,158,Dengue
8,28,Female,1,5.0,1,1.81,129,Dengue
9,30,Male,1,5.0,1,1.44,74,Dengue



Last 10 rows:


,Age,Gender,Fever,Fever_Duration,Skin_Manifestation,WBC,Platelet_Count,Disease
31493,57,Female,0,0.0,0,4830.0,293539,Typhoid
31494,37,Male,0,0.0,0,10036.0,236686,Typhoid
31495,75,Male,0,0.0,0,7153.0,150654,Typhoid
31496,58,Male,0,0.0,1,10066.0,215977,Typhoid
31497,53,Female,0,0.0,0,6720.0,217328,Typhoid
31498,69,Male,0,0.0,0,6008.0,393632,Typhoid
31499,50,Male,1,2.0,0,9809.0,403479,Typhoid
31500,77,Female,0,0.0,0,7362.0,215301,Typhoid
31501,34,Female,1,13.0,0,6032.0,177619,Typhoid
31502,2,Male,0,0.0,0,7875.0,174284,Typhoid



Disease counts:
Disease
Typhoid    31087
Dengue       416
Name: count, dtype: int64

Data types:
Age                     int64
Gender                 object
Fever                   int64
Fever_Duration        float64
Skin_Manifestation      int64
WBC                   float64
Platelet_Count          int64
Disease                object
dtype: object

Missing values:
Age                   0
Gender                0
Fever                 0
Fever_Duration        4
Skin_Manifestation    0
WBC                   1
Platelet_Count        0
Disease               0
dtype: int64


In [7]:
# ============================================
# STEP 6: CORRECT TYPHOID DISEASE LABELS
# ============================================

# Check the original Typhoid status distribution
print("Original Typhoid Status distribution:")
print(typhoid["Typhoid Status"].value_counts(dropna=False))

# Create a copy of our Typhoid feature dataframe
t_corrected = t.copy()

# Keep only confirmed Typhoid categories
confirmed_typhoid = [
    "Acute Typhoid Fever",
    "Relapsing Typhoid",
    "Complicated Typhoid"
]

# Keep confirmed Typhoid cases only
t_confirmed = t_corrected[
    typhoid["Typhoid Status"].isin(confirmed_typhoid)
].copy()

# Assign disease label
t_confirmed["Disease"] = "Typhoid"

print("\nConfirmed Typhoid records:")
print(len(t_confirmed))

print("\nTyphoid status distribution:")
print(
    typhoid.loc[
        t_confirmed.index,
        "Typhoid Status"
    ].value_counts()
)

Original Typhoid Status distribution:
Typhoid Status
Normal or No Typhoid    21701
Acute Typhoid Fever      5649
Relapsing Typhoid        2486
Complicated Typhoid      1251
Name: count, dtype: int64

Confirmed Typhoid records:
9386

Typhoid status distribution:
Typhoid Status
Acute Typhoid Fever    5649
Relapsing Typhoid      2486
Complicated Typhoid    1251
Name: count, dtype: int64


In [8]:
# ============================================
# STEP 7: BUILD CORRECT IDENTIFICATION DATASET
# ============================================

# Dengue dataset
d_corrected = d.copy()
d_corrected["Disease"] = "Dengue"

# Combine Dengue with confirmed Typhoid only
identification_data = pd.concat(
    [d_corrected, t_confirmed],
    ignore_index=True
)

print("Final identification dataset shape:")
print(identification_data.shape)

print("\nDisease distribution:")
print(identification_data["Disease"].value_counts())

print("\nMissing values:")
print(identification_data.isnull().sum())

print("\nFinal columns:")
print(identification_data.columns.tolist())

Final identification dataset shape:
(9802, 8)

Disease distribution:
Disease
Typhoid    9386
Dengue      416
Name: count, dtype: int64

Missing values:
Age                   0
Gender                0
Fever                 0
Fever_Duration        4
Skin_Manifestation    0
WBC                   1
Platelet_Count        0
Disease               0
dtype: int64

Final columns:
['Age', 'Gender', 'Fever', 'Fever_Duration', 'Skin_Manifestation', 'WBC', 'Platelet_Count', 'Disease']


In [9]:
# ============================================
# STEP 8: HANDLE MISSING VALUES
# ============================================

# Check missing values before cleaning
print("Missing values BEFORE:")
print(identification_data.isnull().sum())

# Numeric columns
numeric_cols = [
    "Age",
    "Fever",
    "Fever_Duration",
    "Skin_Manifestation",
    "WBC",
    "Platelet_Count"
]

# Fill numeric missing values using median
for col in numeric_cols:
    identification_data[col] = identification_data[col].fillna(
        identification_data[col].median()
    )

print("\nMissing values AFTER:")
print(identification_data.isnull().sum())

Missing values BEFORE:
Age                   0
Gender                0
Fever                 0
Fever_Duration        4
Skin_Manifestation    0
WBC                   1
Platelet_Count        0
Disease               0
dtype: int64

Missing values AFTER:
Age                   0
Gender                0
Fever                 0
Fever_Duration        0
Skin_Manifestation    0
WBC                   0
Platelet_Count        0
Disease               0
dtype: int64


In [10]:
# ============================================
# STEP 9: CHECK FINAL CLASS DISTRIBUTION
# ============================================

print("Final disease distribution:")
print(identification_data["Disease"].value_counts())

print("\nClass percentages:")
print(
    identification_data["Disease"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Final disease distribution:
Disease
Typhoid    9386
Dengue      416
Name: count, dtype: int64

Class percentages:
Disease
Typhoid    95.76
Dengue      4.24
Name: proportion, dtype: float64


In [11]:
# ============================================
# STEP 10: FINAL DATA QUALITY CHECK
# ============================================

print("Shape:", identification_data.shape)

print("\nData types:")
print(identification_data.dtypes)

print("\nMissing values:")
print(identification_data.isnull().sum())

print("\nDuplicate rows:")
print(identification_data.duplicated().sum())

print("\nSample:")
display(identification_data.head())

Shape: (9802, 8)

Data types:
Age                     int64
Gender                 object
Fever                   int64
Fever_Duration        float64
Skin_Manifestation      int64
WBC                   float64
Platelet_Count          int64
Disease                object
dtype: object

Missing values:
Age                   0
Gender                0
Fever                 0
Fever_Duration        0
Skin_Manifestation    0
WBC                   0
Platelet_Count        0
Disease               0
dtype: int64

Duplicate rows:
0

Sample:


,Age,Gender,Fever,Fever_Duration,Skin_Manifestation,WBC,Platelet_Count,Disease
0,30,Male,1,6.0,0,8.55,50,Dengue
1,30,Male,1,4.0,0,2.48,187,Dengue
2,29,Male,1,7.0,0,3.39,39,Dengue
3,43,Female,1,2.0,0,2.59,50,Dengue
4,27,Male,1,15.0,1,5.60,270,Dengue


In [12]:
# ============================================
# STEP 11: TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

# Features and target
X = identification_data.drop(columns=["Disease"])
y = identification_data["Disease"]

# Stratified split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

print("\nTraining percentages:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting percentages:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training samples: 7841
Testing samples : 1961

Training class distribution:
Disease
Typhoid    7508
Dengue      333
Name: count, dtype: int64

Testing class distribution:
Disease
Typhoid    1878
Dengue       83
Name: count, dtype: int64

Training percentages:
Disease
Typhoid    95.75
Dengue      4.25
Name: proportion, dtype: float64

Testing percentages:
Disease
Typhoid    95.77
Dengue      4.23
Name: proportion, dtype: float64


In [13]:
# ============================================
# STEP 12: PREPROCESSING PIPELINE
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "Age",
    "Fever",
    "Fever_Duration",
    "Skin_Manifestation",
    "WBC",
    "Platelet_Count"
]

categorical_features = [
    "Gender"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary"
            ),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [14]:
# ============================================
# STEP 13: CHECK FEATURE RANGES BY DISEASE
# ============================================

numeric_check = [
    "Age",
    "Fever_Duration",
    "WBC",
    "Platelet_Count"
]

print("=" * 70)
print("NUMERICAL FEATURE RANGES BY DISEASE")
print("=" * 70)

for disease in identification_data["Disease"].unique():

    print(f"\n--- {disease} ---")

    subset = identification_data[
        identification_data["Disease"] == disease
    ]

    print(
        subset[numeric_check].describe()
        .round(2)
    )

NUMERICAL FEATURE RANGES BY DISEASE

--- Dengue ---
          Age  Fever_Duration      WBC  Platelet_Count
count  416.00          416.00   416.00          416.00
mean    35.38            4.62    21.41          109.84
std     12.24            2.37   365.74           56.16
min      1.00            0.25     0.87            6.00
25%     27.00            3.00     2.11           68.75
50%     33.00            4.00     2.84           99.00
75%     43.00            6.00     4.24          140.00
max     76.00           20.00  7463.00          336.00

--- Typhoid ---
           Age  Fever_Duration       WBC  Platelet_Count
count  9386.00         9386.00   9386.00         9386.00
mean     40.52           10.54   8219.51       268719.80
std      23.25            6.29   3816.45       106399.13
min       1.00            0.00   3000.00        50163.00
25%      20.00            5.00   5344.00       186512.25
50%      40.00           11.00   7666.00       272274.00
75%      61.00           16.00   9938

In [15]:
# ============================================
# STEP 13B: CHECK WBC AND PLATELET RANGES
# ============================================

for disease in ["Dengue", "Typhoid"]:

    subset = identification_data[
        identification_data["Disease"] == disease
    ]

    print(f"\n{disease}")

    print("WBC:")
    print(
        "Min =", subset["WBC"].min(),
        "| Max =", subset["WBC"].max()
    )

    print("Platelets:")
    print(
        "Min =", subset["Platelet_Count"].min(),
        "| Max =", subset["Platelet_Count"].max()
    )


Dengue
WBC:
Min = 0.87 | Max = 7463.0
Platelets:
Min = 6 | Max = 336

Typhoid
WBC:
Min = 3000.0 | Max = 19994.0
Platelets:
Min = 50163 | Max = 449994


In [16]:
# ============================================
# STEP 14: INSPECT ORIGINAL COLUMN INFORMATION
# ============================================

print("DENGUE ORIGINAL COLUMNS")
print("=" * 60)

for col in dengue.columns:
    print(repr(col))

print("\nTYPHOID ORIGINAL COLUMNS")
print("=" * 60)

for col in typhoid.columns:
    print(repr(col))

DENGUE ORIGINAL COLUMNS
'Gender'
'Age'
'Where_Infected'
'Year'
'Course_at_admission_(day)'
'Fever'
'Body_Temparature'
'Fever_Duration'
'Rash'
'Course_when_get_Rash'
'Muscle_Pain'
'Bleeding'
'WBC'
'N'
'HB'
'PLT'
'ALT'
'AST'
'BUN'
'Cr'
'CK'
'CK-MB'
'K+'
'Na+'
'C1'
'CD_4+'
'CD_8+'
'CRP'
'PCT'
'Hospitalization'

TYPHOID ORIGINAL COLUMNS
'Age'
'Gender'
'Location'
'Socioeconomic Status'
'Water Source Type'
'Sanitation Facilities'
'Hand Hygiene'
'Consumption of Street Food'
'Fever Duration (Days)'
'Gastrointestinal Symptoms'
'Neurological Symptoms'
'Skin Manifestations'
'Complications'
'White Blood Cell Count'
'Platelet Count'
'Blood Culture Result'
'Widal Test'
'Typhidot Test'
'Typhoid Vaccination Status'
'Previous History of Typhoid'
'Weather Condition'
'Ongoing Infection in Society'
'Typhoid Status'


In [17]:
# ============================================
# STEP 14B: CHECK WBC / PLATELET VALUES
# ============================================

print("DENGUE WBC SAMPLE:")
print(dengue["WBC"].head(20).tolist())

print("\nDENGUE PLT SAMPLE:")
print(dengue["PLT"].head(20).tolist())

print("\nTYPHOID WBC SAMPLE:")
print(typhoid["White Blood Cell Count"].head(20).tolist())

print("\nTYPHOID PLATELET SAMPLE:")
print(typhoid["Platelet Count"].head(20).tolist())

DENGUE WBC SAMPLE:
['8.55', '2.48', '3.39', '2.59', '5.6', '2.1', '1.75', '3.43', '1.81', '1.44', '3.91', '5.11', '1.78', '2.33', '1', '2.7', '1.16', '8.92', '1.87', '1.78']

DENGUE PLT SAMPLE:
[50, 187, 39, 50, 270, 42, 69, 158, 129, 74, 67, 160, 108, 115, 68, 97, 52, 93, 114, 133]

TYPHOID WBC SAMPLE:
[3446, 8002, 16488, 6390, 10399, 9401, 8604, 8171, 5997, 3066, 16439, 6100, 10207, 3660, 4987, 4194, 5418, 6334, 7972, 10473]

TYPHOID PLATELET SAMPLE:
[300334, 158635, 193736, 283273, 241089, 397072, 200887, 269098, 86723, 424607, 415824, 308431, 198839, 298931, 334598, 297266, 307520, 327941, 198703, 187230]


In [18]:
# ============================================
# STEP 15: FINAL COMMON FEATURE SET
# ============================================

final_features = [
    "Age",
    "Gender",
    "Fever",
    "Fever_Duration",
    "Skin_Manifestation"
]

X = identification_data[final_features].copy()
y = identification_data["Disease"].copy()

print("Final input features:")
print(final_features)

print("\nFeature shape:", X.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values:")
print(X.isnull().sum())

Final input features:
['Age', 'Gender', 'Fever', 'Fever_Duration', 'Skin_Manifestation']

Feature shape: (9802, 5)

Target distribution:
Disease
Typhoid    9386
Dengue      416
Name: count, dtype: int64

Missing values:
Age                   0
Gender                0
Fever                 0
Fever_Duration        0
Skin_Manifestation    0
dtype: int64


In [19]:
# ============================================
# STEP 15B: FINAL FEATURE INSPECTION
# ============================================

display(X.head(10))

print("\nFeature data types:")
print(X.dtypes)

print("\nDisease labels:")
print(y.unique())

,Age,Gender,Fever,Fever_Duration,Skin_Manifestation
0,30,Male,1,6.0,0
1,30,Male,1,4.0,0
2,29,Male,1,7.0,0
3,43,Female,1,2.0,0
4,27,Male,1,15.0,1
5,29,Male,1,2.0,1
6,43,Male,1,4.0,0
7,30,Female,1,10.0,1
8,28,Female,1,5.0,1
9,30,Male,1,5.0,1



Feature data types:
Age                     int64
Gender                 object
Fever                   int64
Fever_Duration        float64
Skin_Manifestation      int64
dtype: object

Disease labels:
['Dengue' 'Typhoid']


Step 16 — Create the final stratified split

In [20]:
# ============================================
# STEP 16: FINAL STRATIFIED TRAIN/TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training samples: 7841
Testing samples : 1961

Training distribution:
Disease
Typhoid    7508
Dengue      333
Name: count, dtype: int64

Testing distribution:
Disease
Typhoid    1878
Dengue       83
Name: count, dtype: int64


Step 17 — Build the preprocessing pipeline

In [21]:
# ============================================
# STEP 17: FINAL PREPROCESSING PIPELINE
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "Age",
    "Fever",
    "Fever_Duration",
    "Skin_Manifestation"
]

categorical_features = [
    "Gender"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary"
            ),
            categorical_features
        )
    ]
)

print("Final preprocessing pipeline created.")

Final preprocessing pipeline created.


Step 18 — Handle class imbalance during training

In [22]:
# ============================================
# STEP 18: CHECK CLASS IMBALANCE
# ============================================

print("Training class distribution:")
print(y_train.value_counts())

print("\nTraining class percentages:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training class distribution:
Disease
Typhoid    7508
Dengue      333
Name: count, dtype: int64

Training class percentages:
Disease
Typhoid    95.75
Dengue      4.25
Name: proportion, dtype: float64


Step 19 — Train the first candidate: Logistic Regression

In [23]:
# ============================================
# STEP 19: LOGISTIC REGRESSION
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Train ONLY on training data
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


Step 19 — Train the first candidate: Logistic Regression

In [24]:
# ============================================
# STEP 19: LOGISTIC REGRESSION
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Train ONLY on training data
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [25]:
# ============================================
# STEP 19B: LOGISTIC REGRESSION EVALUATION
# ============================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
    f1_score
)

y_pred_lr = logistic_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, y_pred_lr)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred_lr,
        average="macro"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_lr,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_lr,
        labels=["Dengue", "Typhoid"]
    )
)

Accuracy: 0.717491075981642
Balanced Accuracy: 0.8064398167750876
Macro F1: 0.5204557814000791

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.1208    0.9036    0.2131        83
     Typhoid     0.9940    0.7093    0.8278      1878

    accuracy                         0.7175      1961
   macro avg     0.5574    0.8064    0.5205      1961
weighted avg     0.9571    0.7175    0.8018      1961


Confusion Matrix:
[[  75    8]
 [ 546 1332]]


Next: Random Forest

In [26]:
# ============================================
# STEP 20: RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [27]:
# ============================================
# STEP 20B: RANDOM FOREST EVALUATION
# ============================================

y_pred_rf = random_forest_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, y_pred_rf)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred_rf,
        average="macro"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_rf,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_rf,
        labels=["Dengue", "Typhoid"]
    )
)

Accuracy: 0.9321774604793472
Balanced Accuracy: 0.6479079256322414
Macro F1: 0.6303340576043496

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.2642    0.3373    0.2963        83
     Typhoid     0.9704    0.9585    0.9644      1878

    accuracy                         0.9322      1961
   macro avg     0.6173    0.6479    0.6303      1961
weighted avg     0.9405    0.9322    0.9361      1961


Confusion Matrix:
[[  28   55]
 [  78 1800]]


Step 21 — SVM

In [28]:
# ============================================
# STEP 21: SVM CLASSIFIER
# ============================================

from sklearn.svm import SVC

svm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        class_weight="balanced",
        kernel="rbf",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

print("SVM trained successfully.")

SVM trained successfully.


In [29]:
# ============================================
# STEP 21B: SVM EVALUATION
# ============================================

y_pred_svm = svm_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, y_pred_svm)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred_svm,
        average="macro"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_svm,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_svm,
        labels=["Dengue", "Typhoid"]
    )
)

Accuracy: 0.7955124936257012
Balanced Accuracy: 0.8702060638721019
Macro F1: 0.5817044050349032

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.1660    0.9518    0.2826        83
     Typhoid     0.9973    0.7886    0.8808      1878

    accuracy                         0.7955      1961
   macro avg     0.5816    0.8702    0.5817      1961
weighted avg     0.9621    0.7955    0.8554      1961


Confusion Matrix:
[[  79    4]
 [ 397 1481]]



Step 22 — Find common/compatible features

In [30]:
# ============================================
# STEP 22: FEATURE COMPATIBILITY AUDIT
# ============================================

print("=" * 70)
print("DENGUE COLUMNS")
print("=" * 70)

for col in dengue.columns:
    print("-", col)

print("\n" + "=" * 70)
print("TYPHOID COLUMNS")
print("=" * 70)

for col in typhoid.columns:
    print("-", col)

DENGUE COLUMNS
- Gender
- Age
- Where_Infected
- Year
- Course_at_admission_(day)
- Fever
- Body_Temparature
- Fever_Duration
- Rash
- Course_when_get_Rash
- Muscle_Pain
- Bleeding
- WBC
- N
- HB
- PLT
- ALT
- AST
- BUN
- Cr
- CK
- CK-MB
- K+
- Na+
- C1
- CD_4+
- CD_8+
- CRP
- PCT
- Hospitalization

TYPHOID COLUMNS
- Age
- Gender
- Location
- Socioeconomic Status
- Water Source Type
- Sanitation Facilities
- Hand Hygiene
- Consumption of Street Food
- Fever Duration (Days)
- Gastrointestinal Symptoms
- Neurological Symptoms
- Skin Manifestations
- Complications
- White Blood Cell Count
- Platelet Count
- Blood Culture Result
- Widal Test
- Typhidot Test
- Typhoid Vaccination Status
- Previous History of Typhoid
- Weather Condition
- Ongoing Infection in Society
- Typhoid Status


In [31]:
# ============================================
# STEP 22B: INSPECT COMMON CLINICAL FEATURES
# ============================================

check_dengue = [
    "Gender",
    "Fever",
    "Fever_Duration",
    "Rash",
    "Muscle_Pain",
    "Bleeding"
]

check_typhoid = [
    "Gender",
    "Fever Duration (Days)",
    "Skin Manifestations",
    "Gastrointestinal Symptoms",
    "Neurological Symptoms"
]

print("DENGUE CATEGORICAL/SYMPTOM VALUES")
print("=" * 70)

for col in check_dengue:
    if col in dengue.columns:
        print(f"\n{col}:")
        print(dengue[col].value_counts(dropna=False).head(20))

print("\n\nTYPHOID CATEGORICAL/SYMPTOM VALUES")
print("=" * 70)

for col in check_typhoid:
    if col in typhoid.columns:
        print(f"\n{col}:")
        print(typhoid[col].value_counts(dropna=False).head(20))

DENGUE CATEGORICAL/SYMPTOM VALUES

Gender:
Gender
Male      291
Female    125
Name: count, dtype: int64

Fever:
Fever
1    414
0      2
Name: count, dtype: int64

Fever_Duration:
Fever_Duration
4       81
5       75
3       60
6       59
2       41
7       36
1       29
8       14
9        5
10       4
?        3
15       2
20       1
11       1
14       1
12       1
10+      1
0.5      1
0.25     1
Name: count, dtype: int64

Rash:
Rash
No     223
Yes    193
Name: count, dtype: int64

Muscle_Pain:
Muscle_Pain
Yes    320
No      95
??       1
Name: count, dtype: int64

Bleeding:
Bleeding
0    343
1     73
Name: count, dtype: int64


TYPHOID CATEGORICAL/SYMPTOM VALUES

Gender:
Gender
Male      15580
Female    15507
Name: count, dtype: int64

Fever Duration (Days):
Fever Duration (Days)
0     22106
14      464
9       463
8       450
13      446
7       437
18      436
15      433
17      433
5       433
2       427
10      424
19      423
3       421
21      419
11      417
4       416
2

Step 23 — Build the final common-feature dataset

In [32]:
# ============================================
# STEP 23: FINAL COMMON CLINICAL FEATURES
# ============================================

final_features = [
    "Age",
    "Gender",
    "Fever_Duration",
    "Skin_Manifestation"
]

X_final = identification_data[final_features].copy()
y_final = identification_data["Disease"].copy()

print("Final features:")
print(final_features)

print("\nShape:")
print(X_final.shape)

print("\nDisease distribution:")
print(y_final.value_counts())

print("\nMissing values:")
print(X_final.isnull().sum())

display(X_final.head())

Final features:
['Age', 'Gender', 'Fever_Duration', 'Skin_Manifestation']

Shape:
(9802, 4)

Disease distribution:
Disease
Typhoid    9386
Dengue      416
Name: count, dtype: int64

Missing values:
Age                   0
Gender                0
Fever_Duration        0
Skin_Manifestation    0
dtype: int64


,Age,Gender,Fever_Duration,Skin_Manifestation
0,30,Male,6.0,0
1,30,Male,4.0,0
2,29,Male,7.0,0
3,43,Female,2.0,0
4,27,Male,15.0,1


In [33]:
print(
    X_final["Skin_Manifestation"]
    .value_counts(dropna=False)
)

Skin_Manifestation
0    4924
1    4878
Name: count, dtype: int64


Step 24 — Create the final stratified split

In [34]:
# ============================================
# STEP 24: FINAL TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    random_state=42,
    stratify=y_final
)

print("Training samples:", len(X_train_final))
print("Testing samples :", len(X_test_final))

print("\nTraining distribution:")
print(y_train_final.value_counts())

print("\nTesting distribution:")
print(y_test_final.value_counts())

Training samples: 7841
Testing samples : 1961

Training distribution:
Disease
Typhoid    7508
Dengue      333
Name: count, dtype: int64

Testing distribution:
Disease
Typhoid    1878
Dengue       83
Name: count, dtype: int64


Step 25 — Final preprocessing pipeline

In [35]:
# ============================================
# STEP 25: FINAL PREPROCESSING
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features_final = [
    "Age",
    "Fever_Duration",
    "Skin_Manifestation"
]

categorical_features_final = [
    "Gender"
]

preprocessor_final = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features_final
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary"
            ),
            categorical_features_final
        )
    ]
)

print("Final preprocessing pipeline created successfully.")

Final preprocessing pipeline created successfully.


In [36]:
# ============================================
# STEP 25: FINAL PREPROCESSING
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features_final = [
    "Age",
    "Fever_Duration",
    "Skin_Manifestation"
]

categorical_features_final = [
    "Gender"
]

preprocessor_final = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features_final
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary"
            ),
            categorical_features_final
        )
    ]
)

print("Final preprocessing pipeline created successfully.")

Final preprocessing pipeline created successfully.


Step 26 — Train the final Logistic Regression

In [37]:
# ============================================
# STEP 26: FINAL LOGISTIC REGRESSION
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

final_lr_model = Pipeline([
    ("preprocessor", preprocessor_final),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

final_lr_model.fit(
    X_train_final,
    y_train_final
)

print("Final Logistic Regression trained successfully.")

Final Logistic Regression trained successfully.


In [38]:
# ============================================
# STEP 26B: FINAL LOGISTIC REGRESSION EVALUATION
# ============================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred_lr_final = final_lr_model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_final, y_pred_lr_final))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_final,
        y_pred_lr_final
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test_final,
        y_pred_lr_final,
        average="macro"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_final,
        y_pred_lr_final,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_final,
        y_pred_lr_final,
        labels=["Dengue", "Typhoid"]
    )
)

Accuracy: 0.6914839367669556
Balanced Accuracy: 0.7755879749028061
Macro F1: 0.5007925263940534

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.1081    0.8675    0.1923        83
     Typhoid     0.9915    0.6837    0.8093      1878

    accuracy                         0.6915      1961
   macro avg     0.5498    0.7756    0.5008      1961
weighted avg     0.9541    0.6915    0.7832      1961


Confusion Matrix:
[[  72   11]
 [ 594 1284]]


In [39]:
# ============================================
# STEP 27: FINAL RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

final_rf_model = Pipeline([
    ("preprocessor", preprocessor_final),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

final_rf_model.fit(
    X_train_final,
    y_train_final
)

print("Final Random Forest trained successfully.")

Final Random Forest trained successfully.


In [40]:
# ============================================
# STEP 27B: FINAL RANDOM FOREST EVALUATION
# ============================================

y_pred_rf_final = final_rf_model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_final, y_pred_rf_final))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_final,
        y_pred_rf_final
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test_final,
        y_pred_rf_final,
        average="macro"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_final,
        y_pred_rf_final,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_final,
        y_pred_rf_final,
        labels=["Dengue", "Typhoid"]
    )
)

Accuracy: 0.9331973482916879
Balanced Accuracy: 0.6484404069953937
Macro F1: 0.6321958063984996

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.2692    0.3373    0.2995        83
     Typhoid     0.9704    0.9595    0.9649      1878

    accuracy                         0.9332      1961
   macro avg     0.6198    0.6484    0.6322      1961
weighted avg     0.9407    0.9332    0.9368      1961


Confusion Matrix:
[[  28   55]
 [  76 1802]]


Step 28 — Final SVM

In [41]:
# ============================================
# STEP 28: FINAL SVM
# ============================================

from sklearn.svm import SVC

final_svm_model = Pipeline([
    ("preprocessor", preprocessor_final),
    ("classifier", SVC(
        class_weight="balanced",
        kernel="rbf",
        random_state=42
    ))
])

final_svm_model.fit(
    X_train_final,
    y_train_final
)

print("Final SVM trained successfully.")

Final SVM trained successfully.


In [42]:
# ============================================
# STEP 28B: FINAL SVM EVALUATION
# ============================================

y_pred_svm_final = final_svm_model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_final, y_pred_svm_final))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_final,
        y_pred_svm_final
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test_final,
        y_pred_svm_final,
        average="macro"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_final,
        y_pred_svm_final,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_final,
        y_pred_svm_final,
        labels=["Dengue", "Typhoid"]
    )
)

Accuracy: 0.7970423253442122
Balanced Accuracy: 0.8594890745088982
Macro F1: 0.5804423085193309

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.1642    0.9277    0.2790        83
     Typhoid     0.9960    0.7913    0.8819      1878

    accuracy                         0.7970      1961
   macro avg     0.5801    0.8595    0.5804      1961
weighted avg     0.9608    0.7970    0.8564      1961


Confusion Matrix:
[[  77    6]
 [ 392 1486]]


Step 29 — Detailed SVM validation

In [43]:
# ============================================
# STEP 29: FINAL SVM VALIDATION
# ============================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Predictions already generated
y_pred = y_pred_svm_final

# Probability / decision scores
y_score = final_svm_model.decision_function(X_test_final)

print("=" * 60)
print("FINAL SVM VALIDATION")
print("=" * 60)

# Dengue = positive class
dengue_precision = precision_score(
    y_test_final,
    y_pred,
    pos_label="Dengue"
)

dengue_recall = recall_score(
    y_test_final,
    y_pred,
    pos_label="Dengue"
)

dengue_f1 = f1_score(
    y_test_final,
    y_pred,
    pos_label="Dengue"
)

roc_auc = roc_auc_score(
    (y_test_final == "Dengue").astype(int),
    y_score
)

pr_auc = average_precision_score(
    (y_test_final == "Dengue").astype(int),
    y_score
)

print(f"Dengue Precision : {dengue_precision:.4f}")
print(f"Dengue Recall    : {dengue_recall:.4f}")
print(f"Dengue F1        : {dengue_f1:.4f}")
print(f"ROC-AUC          : {roc_auc:.4f}")
print(f"PR-AUC            : {pr_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_final,
        y_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_final,
        y_pred,
        labels=["Dengue", "Typhoid"]
    )
)

FINAL SVM VALIDATION
Dengue Precision : 0.1642
Dengue Recall    : 0.9277
Dengue F1        : 0.2790
ROC-AUC          : 0.0697
PR-AUC            : 0.0225

Classification Report:
              precision    recall  f1-score   support

      Dengue     0.1642    0.9277    0.2790        83
     Typhoid     0.9960    0.7913    0.8819      1878

    accuracy                         0.7970      1961
   macro avg     0.5801    0.8595    0.5804      1961
weighted avg     0.9608    0.7970    0.8564      1961


Confusion Matrix:
[[  77    6]
 [ 392 1486]]


Step 29B — Correct the AUC calculation

In [44]:

# ============================================
# STEP 29B: CHECK SVM CLASS ORDER
# ============================================

svm_classifier = final_svm_model.named_steps["classifier"]

print("SVM class order:")
print(svm_classifier.classes_)

SVM class order:
['Dengue' 'Typhoid']


In [46]:
# ============================================
# STEP 29C: CORRECT SVM AUC / PR-AUC
# ============================================

from sklearn.metrics import roc_auc_score, average_precision_score

# SVM decision scores
y_score = final_svm_model.decision_function(X_test_final)

# Because sklearn reports classes in this order:
# ['Dengue', 'Typhoid']
#
# We want Dengue to be the positive class.
# Reverse the decision score so higher = Dengue.
dengue_score = -y_score

y_test_binary = (
    y_test_final == "Dengue"
).astype(int)

roc_auc_corrected = roc_auc_score(
    y_test_binary,
    dengue_score
)

pr_auc_corrected = average_precision_score(
    y_test_binary,
    dengue_score
)

print("=" * 60)
print("CORRECTED SVM AUC VALIDATION")
print("=" * 60)

print(f"ROC-AUC : {roc_auc_corrected:.4f}")
print(f"PR-AUC  : {pr_auc_corrected:.4f}")

CORRECTED SVM AUC VALIDATION
ROC-AUC : 0.9303
PR-AUC  : 0.3565


Step 30 — Final model decision

In [47]:
# ============================================
# STEP 30: FINAL MODEL COMPARISON
# ============================================

import pandas as pd

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "SVM"
    ],
    "Accuracy": [
        accuracy_score(y_test_final, y_pred_lr_final),
        accuracy_score(y_test_final, y_pred_rf_final),
        accuracy_score(y_test_final, y_pred_svm_final)
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_test_final, y_pred_lr_final),
        balanced_accuracy_score(y_test_final, y_pred_rf_final),
        balanced_accuracy_score(y_test_final, y_pred_svm_final)
    ],
    "Macro F1": [
        f1_score(y_test_final, y_pred_lr_final, average="macro"),
        f1_score(y_test_final, y_pred_rf_final, average="macro"),
        f1_score(y_test_final, y_pred_svm_final, average="macro")
    ],
    "Dengue Precision": [
        precision_score(y_test_final, y_pred_lr_final, pos_label="Dengue"),
        precision_score(y_test_final, y_pred_rf_final, pos_label="Dengue"),
        precision_score(y_test_final, y_pred_svm_final, pos_label="Dengue")
    ],
    "Dengue Recall": [
        recall_score(y_test_final, y_pred_lr_final, pos_label="Dengue"),
        recall_score(y_test_final, y_pred_rf_final, pos_label="Dengue"),
        recall_score(y_test_final, y_pred_svm_final, pos_label="Dengue")
    ],
    "Dengue F1": [
        f1_score(y_test_final, y_pred_lr_final, pos_label="Dengue"),
        f1_score(y_test_final, y_pred_rf_final, pos_label="Dengue"),
        f1_score(y_test_final, y_pred_svm_final, pos_label="Dengue")
    ]
})

print("=" * 90)
print("FINAL IDENTIFICATION MODEL COMPARISON")
print("=" * 90)

display(
    model_comparison.round(4)
)

FINAL IDENTIFICATION MODEL COMPARISON


,Model,Accuracy,Balanced Accuracy,Macro F1,Dengue Precision,Dengue Recall,Dengue F1
0,Logistic Regression,0.6915,0.7756,0.5008,0.1081,0.8675,0.1923
1,Random Forest,0.9332,0.6484,0.6322,0.2692,0.3373,0.2995
2,SVM,0.7970,0.8595,0.5804,0.1642,0.9277,0.2790


Step 31 — Save the final identification model

In [48]:
# ============================================
# STEP 31: SAVE FINAL IDENTIFICATION MODEL
# ============================================

import joblib

model_path = "disease_identification_svm.pkl"

joblib.dump(
    final_svm_model,
    model_path
)

print("Final identification model saved successfully.")
print("File:", model_path)

Final identification model saved successfully.
File: disease_identification_svm.pkl


In [49]:
# ============================================
# STEP 31B: VERIFY SAVED MODEL
# ============================================

loaded_identification_model = joblib.load(
    "disease_identification_svm.pkl"
)

print("Saved model loaded successfully.")
print(
    "Model type:",
    type(loaded_identification_model)
)

Saved model loaded successfully.
Model type: <class 'sklearn.pipeline.Pipeline'>
